In [73]:
# MTP Co-Intelligent Finance: AI-driven transformations in Capital Market, Stock & Risk Prediction
# By - M24DE3035 - Geetika Vijay
# Python Implementation Results Analysis for Linear Regressions Algorithms


In [74]:
# Stock Prediction with Linear, Ridge, and Lasso Regression
# Using Yahoo Finance data + technical indicators (MA, RSI, MACD)
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score


In [75]:
# ************** 1.Download stock data for ticker Apple Inc. ************** #
ticker = "AAPL"   #
start_date = "2020-01-01"
end_date = "2025-01-01"

df = yf.download(ticker, start=start_date, end=end_date)


/tmp/ipython-input-4239386366.py:6: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed


In [76]:
# ************** 2. Compute technical indicators for further analysis ************** #
# ************** Moving Averages ************** #
df["MA10"] = df["Close"].rolling(window=10).mean()
df["MA50"] = df["Close"].rolling(window=50).mean()

# ************** Relative Strength Index (RSI) ************** #
def compute_RSI(series, window=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

df["RSI14"] = compute_RSI(df["Close"], 14)

# MACD (12-day EMA - 26-day EMA)
df["EMA12"] = df["Close"].ewm(span=12, adjust=False).mean()
df["EMA26"] = df["Close"].ewm(span=26, adjust=False).mean()
df["MACD"] = df["EMA12"] - df["EMA26"]
df["Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()

In [77]:
# ************** 3. Define features and target ************** #
# Features: technical indicators + volume
features = ["MA10", "MA50", "RSI14", "MACD", "Signal", "Volume"]
df = df.dropna()  # drop rows with NaN due to rolling windows

X = df[features]
y = df["Close"].pct_change().shift(-1)
df = df.dropna()

X = df[features]
y = df["Close"].pct_change().shift(-1).dropna()
X = X.iloc[:-1, :]


In [78]:
# ************** 4. Train-test split ************** #
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [79]:
# ************** 5. Linear Regression # ************** #
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
y_pred_lin = lin_reg.predict(X_test)

print("Linear Regression:")
print("Mean Square Error:", mean_squared_error(y_test, y_pred_lin))
print("R2 Score:", r2_score(y_test, y_pred_lin))
print("-" * 40)


Linear Regression:
Mean Square Error: 0.00035648982043668824
R2 Score: -0.005305822197127608
----------------------------------------


In [80]:
# ************** 6. Ridge Regression # ************** #
ridge = Ridge()
ridge_params = {"alpha": [0.01, 0.1, 1, 10, 100]}
ridge_cv = GridSearchCV(ridge, ridge_params, cv=5, scoring="r2")
ridge_cv.fit(X_train, y_train)



/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.45913e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.43277e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.37073e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.29174e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/python3.12/dist-packages/scipy/_lib/_util.py:1233: LinAlgWarning: Ill-conditioned matrix (rcond=1.38703e-17): result may not be accurate.
  return f(*arrays, *other_args, **kwargs)
/usr/local/lib/pytho

GridSearchCV(cv=5, estimator=Ridge(),
             param_grid={'alpha': [0.01, 0.1, 1, 10, 100]}, scoring='r2')

In [81]:
# ************** 6.1 Ridge Regression # ************** #
y_pred_ridge = ridge_cv.predict(X_test)

print("Ridge Regression (best alpha={}):".format(ridge_cv.best_params_["alpha"]))
print("Mean Square Error:", mean_squared_error(y_test, y_pred_ridge))
print("R2 Score:", r2_score(y_test, y_pred_ridge))
print("-" * 40)


Ridge Regression (best alpha=100):
Mean Square Error: 0.00035592103901419523
R2 Score: -0.0037018513603466197
----------------------------------------


In [82]:
# ************** 7. Lasso Regression # ************** #
lasso = Lasso(max_iter=10000)
lasso_params = {"alpha": [0.001, 0.01, 0.1, 1, 10]}
lasso_cv = GridSearchCV(lasso, lasso_params, cv=5, scoring="r2")
lasso_cv.fit(X_train, y_train)

y_pred_lasso = lasso_cv.predict(X_test)

print("Lasso Regression (best alpha={}):".format(lasso_cv.best_params_["alpha"]))
print("Mean Square Error:", mean_squared_error(y_test, y_pred_lasso))
print("R2 Score:", r2_score(y_test, y_pred_lasso))
print("-" * 40)


Lasso Regression (best alpha=0.01):
Mean Square Error: 0.0003557568450100999
R2 Score: -0.003238822183006418
----------------------------------------


In [83]:
# ************** 8. Elastic Net Regression # ************** #

elastic = ElasticNet(max_iter=10000)
elastic_params = {"alpha": [0.001, 0.01, 0.1, 1, 10],
                  "l1_ratio": [0.1, 0.5, 0.7, 0.9]}
elastic_cv = GridSearchCV(elastic, elastic_params, cv=5, scoring="r2")
elastic_cv.fit(X_train, y_train)
y_pred_elastic = elastic_cv.predict(X_test)

In [84]:
# ************** 9. Compare models # ************** #
results = pd.DataFrame({
    "Model": ["Linear", "Ridge", "Lasso", "ElasticNet"],
    "Mean Square Error": [
        mean_squared_error(y_test, y_pred_lin),
        mean_squared_error(y_test, y_pred_ridge),
        mean_squared_error(y_test, y_pred_lasso),
        mean_squared_error(y_test, y_pred_elastic),
    ],
    "R2 Score": [
        r2_score(y_test, y_pred_lin),
        r2_score(y_test, y_pred_ridge),
        r2_score(y_test, y_pred_lasso),
        r2_score(y_test, y_pred_elastic),
    ]
})

print("Model Comparison:\n", results)

Model Comparison:
         Model  Mean Square Error  R2 Score
0      Linear           0.000356 -0.005306
1       Ridge           0.000356 -0.003702
2       Lasso           0.000356 -0.003239
3  ElasticNet           0.000356 -0.003157
